In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [2]:
from sklearn.datasets import load_diabetes

X,y = load_diabetes(return_X_y=True)

In [3]:
x_train,x_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=3)

In [4]:
lr = LinearRegression()

lr.fit(x_train, y_train)


LinearRegression()

In [5]:
y_pred = lr.predict(x_test)

print("R2 score: ", r2_score(y_test, y_pred))

R2 score:  0.4161792211496943


In [6]:
lr.coef_, lr.intercept_

(array([  -1.13744712, -212.8867836 ,  540.45536994,  345.20621542,
        -938.23814645,  516.62060367,  172.85885498,  267.87535242,
         732.63230159,   70.07849485]),
 np.float64(153.13441535285003))

## Batch Gradient Desent

In [7]:
X.shape[0]

442

In [8]:
x_train.shape

(353, 10)

In [9]:
class BatchGD:

  def __init__(self, learning_rate, epoch):
    self.learning_rate = learning_rate
    self.epoch = epoch
    self.intercept = None
    self.coef = None

  def fit(self, x_train, y_train):
    n = x_train.shape[0]   # number of samples
    d = x_train.shape[1]   # number of features

    self.coef = np.ones(d)
    self.intercept = 0

    for i in range(self.epoch):
        y_pred = np.dot(x_train, self.coef) + self.intercept
        error = y_train - y_pred

        slope_int = -2 * np.mean(error)
        slope_coe = -2 * np.dot(x_train.T, error) / n

        self.coef = self.coef - self.learning_rate * slope_coe
        self.intercept = self.intercept - self.learning_rate * slope_int



  def predict(self,x_test):
    return np.dot(x_test, self.coef)+self.intercept

In [10]:
bcd = BatchGD(0.4, 500)

bcd.fit(x_train, y_train)     # must run first
y_pred = bcd.predict(x_test)  # then predict

print("R2 score:", r2_score(y_test, y_pred))

R2 score: 0.3881466672903011


In [11]:
bcd.intercept, bcd.coef

(np.float64(153.191847862373),
 array([  36.38976076,  -68.89248049,  377.78651494,  252.85556801,
         -21.46488348,  -50.84932483, -204.22460678,  151.41643101,
         296.32533293,  157.22502741]))

## Stochastic Gradient Desent

In [13]:
x_train.shape[0]

353

In [26]:
class StochasticGD:

  def __init__(self, learning_rate, epoch):
    self.learning_rate = learning_rate
    self.epoch = epoch
    self.intercept = None
    self.coef = None

  def fit(self, x_train, y_train):
    n, d = x_train.shape
    self.coef = np.ones(d) #Shape (1, d)
    self.intercept = 0

    for i in range(self.epoch):
      for j in range(n):
        idx = np.random.randint(0, x_train.shape[0])
        xi = x_train[idx] # shape (1, d)
        yi = y_train[idx]
        y_pred = np.dot(xi, self.coef) + self.intercept

        error = yi - y_pred

        slope_inte = -2 * error
        self.intercept = self.intercept - self.learning_rate * slope_inte

        slope_coef = -2 * xi * error
        self.coef = self.coef - self.learning_rate * slope_coef

  def predict(self, x_test):
        return np.dot(x_test, self.coef) + self.intercept

In [33]:
sgd = StochasticGD(0.1, 60)

sgd.fit(x_train, y_train)

y_pred = sgd.predict(x_test)

print("R2 score:", r2_score(y_test, y_pred))

R2 score: 0.41747538351482993


In [34]:
sgd.coef, sgd.intercept

(array([ -15.27112408, -225.18712384,  562.97839219,  328.83117422,
        -146.95712369, -106.07943782, -183.03300957,  182.58813127,
         414.52413249,  104.41422279]),
 np.float64(153.59978575518406))

**Using skit learn**

In [45]:
from sklearn.linear_model import SGDRegressor

sgd = SGDRegressor(max_iter=100, learning_rate='constant', eta0=0.01)

sgd.fit(x_train, y_train)
y_pred = sgd.predict(x_test)

print("R2 score:", r2_score(y_test, y_pred))

R2 score: 0.3861393390534914


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_stochastic_gradient.py:1608: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


In [49]:
sgd.coef_, sgd.intercept_

(array([  39.50225128,  -55.75058412,  353.46435195,  238.67769683,
         -11.60059309,  -38.42602401, -197.18497109,  149.98729108,
         280.98206708,  155.53337133]),
 array([149.52307001]))

# Mini Batch Gradient Desent

In [72]:
import random

idx = random.sample(range(0, 11), 10)

[10, 3, 0, 5, 9, 1, 4, 8, 7, 6]


In [75]:
class MiniBatch_GD:

  def __init__(self, batch_size, learning_rate, epoch):
    self.learning_rate = learning_rate
    self.epoch = epoch
    self.intercept = None
    self.coef = None
    self.batch_size = batch_size

  def fit(self, x_train, y_train):
    d,n = x_train.shape
    self.intercept = 0

    self.coef = np.ones(n)

    for i in range(self.epoch):
      for i in range(int(d/self.batch_size)):
        idx = random.sample(range(0, d), self.batch_size)

        xi = x_train[idx]
        yi = y_train[idx]

        y_pred = np.dot(xi, self.coef) + self.intercept
        error = yi-y_pred

        slope_inter = -2 * np.mean(error)
        self.intercept = self.intercept - self.learning_rate*slope_inter

        slope_coef = -2*np.dot(xi, error)
        self.coef = self.coef - self.learning_rate*slope_coef

  def predict(self, x_test):
        return np.dot(x_test, self.coef) + self.intercept

In [78]:
mgd = MiniBatch_GD(10, 0.1, 60)

mgd.fit(x_train, y_train)

y_pred = mgd.predict(x_test)

print("R2 score:", r2_score(y_test, y_pred))

R2 score: 0.4201926641549204
